# Train Model
Loads the `.npy` dataset produced by `create_dataset.ipynb`, defines the GAN (Generator/Discriminator), trains it, and runs the inference/evaluation steps.

In [9]:
import os
import numpy as np
import librosa
import scipy.fft as fft
import soundfile as sf
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from tqdm import tqdm

## 1. Load dataset arrays saved already

In [10]:
data_dir = "dataset_files"

train_original = np.load(os.path.join(data_dir, "train_original.npy"))
train_ciphered = np.load(os.path.join(data_dir, "train_ciphered.npy"))
train_frequencies = np.load(os.path.join(data_dir, "train_frequencies.npy"))

test_original = np.load(os.path.join(data_dir, "test_original.npy"))
test_ciphered = np.load(os.path.join(data_dir, "test_ciphered.npy"))
test_frequencies = np.load(os.path.join(data_dir, "test_frequencies.npy"))

base_frequencies = np.load(os.path.join(data_dir, "base_frequencies.npy"))
audio_paths = np.load(os.path.join(data_dir, "audio_paths.npy"), allow_pickle=True).tolist()

rfft_size = len(base_frequencies)

## 2. Dataset / DataLoader

In [11]:
class AudioDataset(Dataset):
    def __init__(self, frequencies, mag_act, mag_permut):
        self.frequency = frequencies
        self.mag_act = mag_act
        self.mag_permut = mag_permut

    def __len__(self):
        return len(self.frequency)

    def __getitem__(self, idx):
        return [self.frequency[idx], self.mag_act[idx], self.mag_permut[idx]]

train_dataset = AudioDataset(train_frequencies, train_original, train_ciphered)
test_dataset = AudioDataset(test_frequencies, test_original, test_ciphered)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 3. GAN: Generator & Discriminator

In [12]:
class Generator(nn.Module):
    def __init__(self, rfft_size):
        super().__init__() 
        self.lstm = nn.LSTM(
            input_size=rfft_size, hidden_size=512, num_layers=2,
            bidirectional=True, batch_first=True
        )
        self.fc1 = nn.Linear(1024, 2048)  
        self.fc2 = nn.Linear(2048, rfft_size) 
        self.relu = nn.ReLU()
        self.batchnorm1 = nn.BatchNorm1d(2048)
    
    def forward(self, x):
        if x.dim() == 2: x = x.unsqueeze(1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x) 
        return x

class Discriminator(nn.Module):
    def __init__(self, rfft_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=rfft_size, hidden_size=512, num_layers=2,
            bidirectional=True, batch_first=True
        )
        self.fc1 = nn.Linear(1024, 512) 
        self.fc2 = nn.Linear(512, 1) 
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        if x.dim() == 2: x = x.unsqueeze(1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x) 
        return x

## 4. Training setup

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

netG = Generator(rfft_size=rfft_size).to(device)
netD = Discriminator(rfft_size=rfft_size).to(device)

criterion_GAN = nn.BCELoss()
criterion_Recon = nn.L1Loss()  

optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))

lambda_recon = 100  
num_epochs = 50

Training on: cuda


## 5. Training loop

In [14]:
for epoch in tqdm(range(num_epochs), desc="Training"):
    for i, (frequencies, real_audio, encrypted_audio) in enumerate(train_loader):
        
        encrypted_audio = encrypted_audio.to(device)
        real_audio = real_audio.to(device)
        batch_size = encrypted_audio.size(0)
        
        real_labels = torch.ones(batch_size, 1).to(device)
        fake_labels = torch.zeros(batch_size, 1).to(device)
        
        # --- Train Discriminator ---
        netD.zero_grad()
        output_real = netD(real_audio)
        lossD_real = criterion_GAN(output_real, real_labels)
        
        fake_audio = netG(encrypted_audio)
        output_fake = netD(fake_audio.detach()) 
        lossD_fake = criterion_GAN(output_fake, fake_labels)
        
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()

        # --- Train Generator ---
        netG.zero_grad()
        output_fake_for_G = netD(fake_audio)
        lossG_GAN = criterion_GAN(output_fake_for_G, real_labels)
        lossG_Recon = criterion_Recon(fake_audio, real_audio)
        
        lossG = lossG_GAN + (lambda_recon * lossG_Recon)
        lossG.backward()
        optimizerG.step()
        
    # Print status every 2 epochs
    if epoch % 2 == 0:
        print(f"[Epoch {epoch}/{num_epochs}] [D loss: {lossD.item():.4f}] "
              f"[G loss: {lossG.item():.4f}] [L1 Error: {lossG_Recon.item():.4f}]")

Training:   2%|▏         | 1/50 [00:17<14:07, 17.29s/it]

[Epoch 0/50] [D loss: 0.7363] [G loss: 274.3334] [L1 Error: 2.7076]


Training:   6%|▌         | 3/50 [00:51<13:15, 16.93s/it]

[Epoch 2/50] [D loss: 0.5158] [G loss: 258.2647] [L1 Error: 2.5677]


Training:  10%|█         | 5/50 [01:25<12:46, 17.04s/it]

...

Training:  94%|█████████▍| 47/50 [13:28<00:51, 17.19s/it]

[Epoch 46/50] [D loss: 0.8579] [G loss: 213.4862] [L1 Error: 2.1153]


Training:  98%|█████████▊| 49/50 [14:02<00:17, 17.20s/it]

[Epoch 48/50] [D loss: 0.6766] [G loss: 220.2241] [L1 Error: 2.1777]


Training: 100%|██████████| 50/50 [14:19<00:00, 17.19s/it]


## 6. Inference & full-audio evaluation

In [15]:
print("\nRunning Full Audio Evaluation...")

# The same cipher used to build the dataset - needed here to encrypt the raw
# test audio again for the full-audio inference demo below.
class cipher:
    def __init__(self, n):
        self.n = n
    
    def division(self, magnitudes):
        length = len(magnitudes)
        sub_magnitudes = [magnitudes[x*(length//self.n):(x+1)*(length//self.n)] for x in range(self.n)]
        permut_rand = np.random.permutation(self.n)
        permut_magnitudes = [sub_magnitudes[x] for x in permut_rand]
        net_permut_mag = np.concatenate(permut_magnitudes)
        return net_permut_mag, permut_rand

my_cipher = cipher(20)

chunk_size = 25000

test_file = audio_paths[0] 
y, sr = librosa.load(test_file, sr=17000)
N = len(y)

audio_parts = np.array([y[x:x+chunk_size] for x in range(0, N, chunk_size) if x+chunk_size <= N])
fourier_transforms = np.array([fft.rfft(part) for part in audio_parts])
true_mags = np.abs(fourier_transforms)[:, :-1] 

encrypted_mags = np.array([my_cipher.division(chunk)[0] for chunk in true_mags])

netG.eval() 
with torch.no_grad():
    enc_tensor = torch.tensor(encrypted_mags, dtype=torch.float32).to(device)
    decrypted_tensor = netG(enc_tensor)
    decrypted_mags = decrypted_tensor.cpu().numpy()
netG.train() 

def reconstruct_with_phase(mags, original_complex_fft):
    true_phase = np.angle(original_complex_fft)
    padded_mags = np.pad(mags, ((0,0), (0,1)), mode='constant')
    rebuilt_complex = padded_mags * np.exp(1j * true_phase)
    waves = fft.irfft(rebuilt_complex, axis=1)
    return waves.flatten()

wave_real = reconstruct_with_phase(true_mags, fourier_transforms)
wave_enc = reconstruct_with_phase(encrypted_mags, fourier_transforms)
wave_dec = reconstruct_with_phase(decrypted_mags, fourier_transforms)

save_dir = r"C:\Users\yashd\Stuff\Encryption_Decryption-SOC-26\FINAL"
os.makedirs(save_dir, exist_ok=True)

sf.write(os.path.join(save_dir, '1_FULL_original.wav'), wave_real, 17000)
sf.write(os.path.join(save_dir, '2_FULL_encrypted.wav'), wave_enc, 17000)
sf.write(os.path.join(save_dir, '3_FULL_decrypted.wav'), wave_dec, 17000)

# Save the initial (true) vs final (decrypted) magnitude spectra so
# plot_frequency_distribution.ipynb can visualize them
np.save(os.path.join(data_dir, "true_mags.npy"), true_mags)
np.save(os.path.join(data_dir, "decrypted_mags.npy"), decrypted_mags)


Running Full Audio Evaluation...


c:\Users\yashd\Stuff\Encryption_Decryption-SOC-26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7. Evaluation on unseen test dataset

In [16]:
print("\n--- Running Evaluation on Unseen Test Dataset ---")

netG.eval()
netD.eval()

total_l1_loss = 0.0
total_gan_loss = 0.0
total_similarity = 0.0

d_correct_real = 0
d_correct_fake = 0
g_fooled_d = 0
total_samples = 0

num_test_batches = len(test_loader)

with torch.no_grad():
    for i, (frequencies, real_audio, encrypted_audio) in enumerate(test_loader):
        
        encrypted_audio = encrypted_audio.to(device)
        real_audio = real_audio.to(device)
        batch_size = encrypted_audio.size(0)
        total_samples += batch_size
        
        
        fake_audio = netG(encrypted_audio)
        
        
        l1_error = criterion_Recon(fake_audio, real_audio)
        total_l1_loss += l1_error.item()
        
        
        real_flat = real_audio.view(batch_size, -1)
        fake_flat = fake_audio.view(batch_size, -1)
        cos_sim = F.cosine_similarity(fake_flat, real_flat, dim=1)
        total_similarity += cos_sim.mean().item() * 100
        
        
        output_real = netD(real_audio)
        output_fake = netD(fake_audio)
        
        real_labels = torch.ones(batch_size, 1).to(device)
        gan_loss = criterion_GAN(output_fake, real_labels)
        total_gan_loss += gan_loss.item()
        
        
        preds_real = (output_real > 0.5).float()
        preds_fake = (output_fake > 0.5).float()
        
        d_correct_real += (preds_real == 1).sum().item()
        d_correct_fake += (preds_fake == 0).sum().item()
        g_fooled_d += (preds_fake == 1).sum().item()


avg_l1 = total_l1_loss / num_test_batches
avg_gan = total_gan_loss / num_test_batches
avg_similarity = total_similarity / num_test_batches

d_accuracy = (d_correct_real + d_correct_fake) / (total_samples * 2) * 100
g_fooling_accuracy = (g_fooled_d / total_samples) * 100

print(f"Decryption Accuracy (Similarity to Original): {avg_similarity:.2f}%")
print(f"Average L1 Error (Reconstruction Distance):   {avg_l1:.4f}")


--- Running Evaluation on Unseen Test Dataset ---
Decryption Accuracy (Similarity to Original): 75.78%
Average L1 Error (Reconstruction Distance):   2.4815
